In [1]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path(".").resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd

from src.application.contracts import PipelineRequest
from src.application.pipeline import ViRAGEPipeline
from src.application.settings import ViRAGESettings
from rich import print as rprint

In [22]:
tmp_path = Path("../examples").resolve()

In [5]:
from langchain_ollama import ChatOllama

LLM_MODEL = "gemma3:27b-cloud"  # или "llama3.2:1b"
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0,
)

In [9]:
data_path = (tmp_path / "demo.csv").resolve()
pd.DataFrame(
    {
        "date": ["2024-01-01", "2024-01-02", "2024-01-03", "2024-01-04"],
        "sales": [10, 14, 13, 18],
        "region": ["A", "A", "B", "B"],
    }
).to_csv(data_path, index=False)

pipeline = ViRAGEPipeline(settings=ViRAGESettings(artifact_root=tmp_path / "artifacts"),
                          reasoning_llm=llm,
                          spec_llm=llm,
                          vlm=llm,
                          vision_judge_llm=llm
                          )
result = pipeline.invoke(PipelineRequest(query="Show the sales trend over time", data_path=data_path.as_posix()))

In [26]:
rprint(result)

PipelineResult(
    run_id='b9ab9421502a4dfa8fdfe0592213fcad',
    query='Show the sales trend over time',
    data_path='D:/programming/projects/mas_rag/notebooks/../examples/demo.csv',
    query_understanding=QueryUnderstandingResult(
        intent='Show sales trend over time',
        requested_operations=['trend analysis', 'time series visualization'],
        candidate_charts=['line', 'area'],
        constraints=[],
        confidence=0.85,
        task_type='descriptive_analytics',
        user_goal='understand the data visually',
        analysis_goal='identify trends in sales data',
        query_variants=[
            QueryVariant(kind='canonical', text='Show the sales trend over time', confidence=0.9, source='llm'),
            QueryVariant(
                kind='schema_grounding',
                text='Visualize sales as a function of time',
                confidence=0.75,
                source='llm'
            ),
            QueryVariant(
                kind='spec_retrieval',
                text='Display a time series chart of sales',
                confidence=0.7,
                source='llm'
            ),
            QueryVariant(
                kind='analysis',
                text='What is the sales trend over time?',
                confidence=0.65,
                source='llm'
            )
        ],
        ambiguity_notes=[]
    ),
    query_intent_bundle=QueryIntentBundle(
        intent='Show sales trend over time',
        requested_operations=['trend analysis', 'time series visualization'],
        constraints=[],
        task_type='descriptive_analytics',
        user_goal='understand the data visually',
        analysis_goal='identify trends in sales data',
        confidence=0.85,
        query_variants=[
            QueryVariant(kind='canonical', text='Show the sales trend over time', confidence=0.9, source='llm'),
            QueryVariant(
                kind='schema_grounding',
                text='Visualize sales as a function of time',
                confidence=0.75,
                source='llm'
            ),
            QueryVariant(
                kind='spec_retrieval',
                text='Display a time series chart of sales',
                confidence=0.7,
                source='llm'
            ),
            QueryVariant(
                kind='analysis',
                text='What is the sales trend over time?',
                confidence=0.65,
                source='llm'
            )
        ],
        ambiguity_notes=[]
    ),
    request_analysis=RequestAnalysisResult(
        grounded_fields=['date', 'sales'],
        ambiguity_report=[],
        selected_fields=['date', 'sales'],
        normalization_hints=[],
        mappings=[
            RequestFieldMapping(
                query_term='sales trend',
                column_name='sales',
                confidence=0.9,
                rationale="The request explicitly asks for 'sales trend', which directly corresponds to the 'sales'
column."
            ),
            RequestFieldMapping(
                query_term='time',
                column_name='date',
                confidence=0.9,
                rationale="The request asks for trend 'over time', which is represented by the 'date' column."
            )
        ],
        missing_fields=[],
        confidence=0.85
    ),
    planning=PlanningResult(
        steps=[
            PlanningStep(
                name='use_the_provided_visualization_plan_to_generate_a_line_chart',
                description='Use the provided visualization plan to generate a line chart.'
            ),
            PlanningStep(
                name="map_'date'_to_the_x_axis_and_'sales'_to_the_y_axis.",
                description="Map 'date' to the x-axis and 'sales' to the y-axis."
            ),
            PlanningStep(
                name="optionally,_map_'region'_to_the_color_channel_for_segmentati",
                description="Optionally, map 'region' to the color cha